<a href="https://colab.research.google.com/github/arpitraj18/dominance-in-f1/blob/main/dominance_in_f1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import os
%matplotlib inline
pd.set_option('display.max_columns', 100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def import_all():
  data={}
  datapath='/content/drive/MyDrive/F1-Data'
  for dirname, _, filenames in os.walk(datapath):
    for filename in filenames:
        if filename.endswith('.csv'):
            name = filename.replace('.csv', '')
            data[name] = pd.read_csv(os.path.join(dirname, filename))
  return data

In [ ]:
def add_ids(data, key):
  df=data[key]
  n_lines = df.shape[0]
  print(f"\n--- Processing DataFrame: '{key}' ---")
  print(f"Initial DataFrame shape: {df.shape}")

  df = pd.merge(df, data['races'][['raceId',
                                     'year', 'round',
                                     'circuitId', 'date', 'time']],
                  on='raceId', how='left')
  print(f"DataFrame shape after merging 'races' on raceId: {df.shape}")
  if df.shape[0] != n_lines: # Checks if the number of rows changed after the merge
        raise ValueError('Merging raceId went wrong') # Raises an error if the row count is inconsistent

  df = pd.merge(df, data['circuits'][['circuitId',
                                        'circuitRef', 'location', 'country']],
                  on='circuitId', how='left') # Performs a left merge with the 'circuits' DataFrame on 'circuitId'
  print(f"DataFrame shape after merging 'circuits' on circuitId: {df.shape}") # Print shape after merge
  if df.shape[0] != n_lines: # Checks if the number of rows changed after the merge
        raise ValueError('Merging circuitId went wrong') # Raises an error if the row count is inconsistent

  df = pd.merge(df, data['drivers'][['driverId',
                                       'driverRef', 'forename', 'surname',
                                       'dob', 'nationality']].rename(columns={'nationality': 'drv_nat'}),
                  on='driverId', how='left') # Performs a left merge with the 'drivers' DataFrame on 'driverId', renaming 'nationality' column
  print(f"DataFrame shape after merging 'drivers' on driverId: {df.shape}") # Print shape after merge
  if df.shape[0] != n_lines: # Checks if the number of rows changed after the merge
        raise ValueError('Merging driverId went wrong') # Raises an error if the row count is inconsistent

  if (key != 'lap_times') and (key != 'pit_stops'): # Checks if the current DataFrame is not 'lap_times' or 'pit_stops'
        df = pd.merge(df, data['constructors'][['constructorId',
                                                'constructorRef',
                                                'name', 'nationality']].rename(columns={'nationality': 'cstr_nat'}),
                      on='constructorId', how='left') # Performs a left merge with 'constructors' DataFrame on 'constructorId', renaming 'nationality'
        print(f"DataFrame shape after merging 'constructors' on constructorId: {df.shape}") # Print shape after merge
        if df.shape[0] != n_lines: # Checks if the number of rows changed after the merge
            raise ValueError('Merging constructorId went wrong') # Raises an error if the row count is inconsistent

  if key == 'results': # Checks if the current DataFrame is 'results'
        df = pd.merge(df, data['status'],
                      on='statusId', how='left') # Performs a left merge with the 'status' DataFrame on 'statusId'
        print(f"DataFrame shape after merging 'status' on statusId: {df.shape}") # Print shape after merge
        if df.shape[0] != n_lines: # Checks if the number of rows changed after the merge
            raise ValueError('Merging statusId went wrong') # Raises an error if the row count is inconsistent
  display(df.head())
  return df # Returns the merged DataFrame


In [ ]:

data = import_all()

res = add_ids(data, 'results')
qual = add_ids(data, 'qualifying')
laps = add_ids(data, 'lap_times')
pits = add_ids(data, 'pit_stops')

laps.rename(columns={'time_x': 'lap_time', 'time_y': 'time'}, inplace=True)
res.rename(columns={'time_x': 'race_time', 'time_y': 'time'}, inplace=True)
pits.rename(columns={'time_x': 'pit_time', 'time_y': 'time'}, inplace=True)

laps = pd.merge(laps, res[['raceId', 'driverId',
                           'constructorRef', 'name', 'cstr_nat']],
                on=['raceId', 'driverId'], how='left')
pits = pd.merge(pits, res[['raceId', 'driverId',
                           'constructorRef', 'name', 'cstr_nat']],
                on=['raceId', 'driverId'], how='left')


--- Processing DataFrame: 'results' ---
Initial DataFrame shape: (26759, 18)
DataFrame shape after merging 'races' on raceId: (26759, 23)
DataFrame shape after merging 'circuits' on circuitId: (26759, 26)
DataFrame shape after merging 'drivers' on driverId: (26759, 31)
DataFrame shape after merging 'constructors' on constructorId: (26759, 34)
DataFrame shape after merging 'status' on statusId: (26759, 35)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time_x,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,date,time_y,circuitRef,location,country,driverRef,forename,surname,dob,drv_nat,constructorRef,name,cstr_nat,status
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,hamilton,Lewis,Hamilton,1985-01-07,British,mclaren,McLaren,British,Finished
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,heidfeld,Nick,Heidfeld,1977-05-10,German,bmw_sauber,BMW Sauber,German,Finished
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,rosberg,Nico,Rosberg,1985-06-27,German,williams,Williams,British,Finished
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,alonso,Fernando,Alonso,1981-07-29,Spanish,renault,Renault,French,Finished
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,kovalainen,Heikki,Kovalainen,1981-10-19,Finnish,mclaren,McLaren,British,Finished



--- Processing DataFrame: 'qualifying' ---
Initial DataFrame shape: (10494, 9)
DataFrame shape after merging 'races' on raceId: (10494, 14)
DataFrame shape after merging 'circuits' on circuitId: (10494, 17)
DataFrame shape after merging 'drivers' on driverId: (10494, 22)
DataFrame shape after merging 'constructors' on constructorId: (10494, 25)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,date,time,circuitRef,location,country,driverRef,forename,surname,dob,drv_nat,constructorRef,name,cstr_nat
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,hamilton,Lewis,Hamilton,1985-01-07,British,mclaren,McLaren,British
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,kubica,Robert,Kubica,1984-12-07,Polish,bmw_sauber,BMW Sauber,German
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,kovalainen,Heikki,Kovalainen,1981-10-19,Finnish,mclaren,McLaren,British
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,massa,Felipe,Massa,1981-04-25,Brazilian,ferrari,Ferrari,Italian
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236,2008,1,1,2008-03-16,04:30:00,albert_park,Melbourne,Australia,heidfeld,Nick,Heidfeld,1977-05-10,German,bmw_sauber,BMW Sauber,German



--- Processing DataFrame: 'lap_times' ---
Initial DataFrame shape: (589081, 6)
DataFrame shape after merging 'races' on raceId: (589081, 11)
DataFrame shape after merging 'circuits' on circuitId: (589081, 14)
DataFrame shape after merging 'drivers' on driverId: (589081, 19)


,raceId,driverId,lap,position,time_x,milliseconds,year,round,circuitId,date,time_y,circuitRef,location,country,driverRef,forename,surname,dob,drv_nat
0,841,20,1,1,1:38.109,98109,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,vettel,Sebastian,Vettel,1987-07-03,German
1,841,20,2,1,1:33.006,93006,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,vettel,Sebastian,Vettel,1987-07-03,German
2,841,20,3,1,1:32.713,92713,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,vettel,Sebastian,Vettel,1987-07-03,German
3,841,20,4,1,1:32.803,92803,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,vettel,Sebastian,Vettel,1987-07-03,German
4,841,20,5,1,1:32.342,92342,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,vettel,Sebastian,Vettel,1987-07-03,German



--- Processing DataFrame: 'pit_stops' ---
Initial DataFrame shape: (11371, 7)
DataFrame shape after merging 'races' on raceId: (11371, 12)
DataFrame shape after merging 'circuits' on circuitId: (11371, 15)
DataFrame shape after merging 'drivers' on driverId: (11371, 20)


,raceId,driverId,stop,lap,time_x,duration,milliseconds,year,round,circuitId,date,time_y,circuitRef,location,country,driverRef,forename,surname,dob,drv_nat
0,841,153,1,1,17:05:23,26.898,26898,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,alguersuari,Jaime,Alguersuari,1990-03-23,Spanish
1,841,30,1,1,17:05:52,25.021,25021,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,michael_schumacher,Michael,Schumacher,1969-01-03,German
2,841,17,1,11,17:20:48,23.426,23426,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,webber,Mark,Webber,1976-08-27,Australian
3,841,4,1,12,17:22:34,23.251,23251,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,alonso,Fernando,Alonso,1981-07-29,Spanish
4,841,13,1,13,17:24:10,23.842,23842,2011,1,1,2011-03-27,06:00:00,albert_park,Melbourne,Australia,massa,Felipe,Massa,1981-04-25,Brazilian
